# 11 — RQ2 collect comments

Harvests the RQ2 cross-community corpus via the YouTube Data API v3, following the same procedure as RQ1 (notebook 01). Four community groups × four channels (16 total). Eligible videos are public, longer than 5 minutes, have ≥1 comment, and were published in a single 12-month window (19 Aug 2025 → 19 Aug 2026). Up to 20 videos per channel are sampled by recency (no titles/keywords); up to 200 newest top-level comments are retrieved per video, **excluding replies**. Identical comments with different IDs are kept as separate occurrences.

Needs `YOUTUBE_API_KEY` in `.env`. Writes the raw corpus to `data/raw/rq2/`.

> Because collection uses the live API, an exact re-run will not reproduce the frozen corpus behind the reported results; the scored corpus is shipped for verification. This notebook documents the collection method.

In [ ]:
import os
import time
import re
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from dotenv import dotenv_values
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ---- paths & key ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "rq2"
RAW_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = dotenv_values(PROJECT_ROOT / ".env").get("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY missing from .env (see .env.example)")

# ---- fixed collection parameters (match RQ1) ----
WINDOW_START = datetime(2025, 8, 19, tzinfo=timezone.utc)
WINDOW_END   = datetime(2026, 8, 19, tzinfo=timezone.utc)   # exclusive
MIN_VIDEO_SECONDS      = 5 * 60
MAX_VIDEOS_PER_CHANNEL = 20
MAX_COMMENTS_PER_VIDEO = 200
COMMENT_ORDER          = "time"

# ---- the 16 channels: four community groups x four channels ----
CHANNELS = [
    {"handle": "@SandmanMGTOW",         "community_group": "manosphere_adjacent"},
    {"handle": "@manguide",             "community_group": "manosphere_adjacent"},
    {"handle": "@TheChowTimePod",       "community_group": "manosphere_adjacent"},
    {"handle": "@ItsComplicatedChannel","community_group": "manosphere_adjacent"},

    {"handle": "@ForTheDadsPod",        "community_group": "male_oriented_wellbeing"},
    {"handle": "@ManTalks",             "community_group": "male_oriented_wellbeing"},
    {"handle": "@ForPurposeDrivenMen",  "community_group": "male_oriented_wellbeing"},
    {"handle": "@menshealthmag",        "community_group": "male_oriented_wellbeing"},

    {"handle": "@aliabdaal",            "community_group": "mainstream_self_improvement"},
    {"handle": "@melrobbins",           "community_group": "mainstream_self_improvement"},
    {"handle": "@hubermanlab",          "community_group": "mainstream_self_improvement"},
    {"handle": "@richroll",             "community_group": "mainstream_self_improvement"},

    {"handle": "@JamieOliver",          "community_group": "unrelated_control"},
    {"handle": "@laura88lee",           "community_group": "unrelated_control"},
    {"handle": "@LoneFox",              "community_group": "unrelated_control"},
    {"handle": "@HuwRichards",          "community_group": "unrelated_control"},
]
print(f"{len(CHANNELS)} channels across "
      f"{len({c['community_group'] for c in CHANNELS})} community groups")


In [ ]:
youtube = build("youtube", "v3", developerKey=API_KEY)

_DUR = re.compile(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?")
def duration_seconds(iso):
    m = _DUR.fullmatch(iso or "")
    if not m:
        return 0
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + mi * 60 + s

def resolve_channel(handle):
    r = youtube.channels().list(part="snippet,contentDetails", forHandle=handle).execute()
    items = r.get("items", [])
    if not items:
        raise ValueError(f"Could not resolve {handle}")
    it = items[0]
    return {
        "channel_id": it["id"],
        "channel_name": it["snippet"]["title"],
        "uploads_playlist": it["contentDetails"]["relatedPlaylists"]["uploads"],
    }

def list_upload_video_ids(uploads_playlist):
    ids, token = [], None
    while True:
        r = youtube.playlistItems().list(
            part="contentDetails", playlistId=uploads_playlist,
            maxResults=50, pageToken=token).execute()
        for it in r.get("items", []):
            ids.append(it["contentDetails"]["videoId"])
        token = r.get("nextPageToken")
        if not token:
            return ids

def video_details(video_ids):
    out = []
    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i + 50]
        r = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(chunk)).execute()
        for v in r.get("items", []):
            out.append({
                "video_id": v["id"],
                "video_title": v["snippet"]["title"],
                "published": datetime.fromisoformat(
                    v["snippet"]["publishedAt"].replace("Z", "+00:00")),
                "duration_s": duration_seconds(v["contentDetails"].get("duration")),
                "comment_count": int(v.get("statistics", {}).get("commentCount", 0)),
                "privacy": v.get("status", {}).get("privacyStatus", ""),
            })
    return out

def eligible_videos(channel):
    """Public, >=5 min, >=1 comment, published in the window; newest 20."""
    vids = video_details(list_upload_video_ids(channel["uploads_playlist"]))
    ok = [v for v in vids
          if v["privacy"] == "public"
          and v["duration_s"] >= MIN_VIDEO_SECONDS
          and v["comment_count"] >= 1
          and WINDOW_START <= v["published"] < WINDOW_END]
    ok.sort(key=lambda v: v["published"], reverse=True)
    return ok[:MAX_VIDEOS_PER_CHANNEL]

def top_level_comments(video_id, limit=MAX_COMMENTS_PER_VIDEO):
    """Up to `limit` newest TOP-LEVEL comments (replies excluded)."""
    rows, token = [], None
    while len(rows) < limit:
        try:
            r = youtube.commentThreads().list(
                part="snippet", videoId=video_id,
                maxResults=min(100, limit - len(rows)),
                order=COMMENT_ORDER, textFormat="plainText",
                pageToken=token).execute()
        except HttpError:
            break   # comments disabled / unavailable
        for it in r.get("items", []):
            c = it["snippet"]["topLevelComment"]["snippet"]
            rows.append({
                "comment_id": it["snippet"]["topLevelComment"]["id"],
                "text": c.get("textDisplay", ""),
                "author": c.get("authorDisplayName", ""),
                "comment_published": c.get("publishedAt", ""),
                "like_count": c.get("likeCount", 0),
                "is_reply": False,
            })
        token = r.get("nextPageToken")
        if not token:
            break
    return rows[:limit]


In [ ]:
# ---- harvest ----
COMMENTS_FILE = RAW_DIR / "rq2_raw_comments.csv"
VIDEOS_FILE   = RAW_DIR / "rq2_selected_videos.csv"

all_comments, all_videos = [], []
for i, ch in enumerate(CHANNELS, 1):
    resolved = resolve_channel(ch["handle"])
    print(f"[{i:>2}/{len(CHANNELS)}] {ch['handle']:<22} -> {resolved['channel_name']}")
    vids = eligible_videos(resolved)
    print(f"      {len(vids)} eligible videos in window")
    for v in vids:
        all_videos.append({
            "community_group": ch["community_group"],
            "channel_handle": ch["handle"],
            "channel_name": resolved["channel_name"],
            "video_id": v["video_id"],
            "video_title": v["video_title"],
            "video_published": v["published"].isoformat(),
        })
        for c in top_level_comments(v["video_id"]):
            all_comments.append({
                "community_group": ch["community_group"],
                "channel_handle": ch["handle"],
                "channel_name": resolved["channel_name"],
                "video_id": v["video_id"],
                "video_title": v["video_title"],
                **c,
            })
    # checkpoint after each channel
    pd.DataFrame(all_comments).to_csv(COMMENTS_FILE, index=False, encoding="utf-8-sig")
    pd.DataFrame(all_videos).to_csv(VIDEOS_FILE, index=False, encoding="utf-8-sig")
    time.sleep(0.5)

comments_df = pd.DataFrame(all_comments)
print("\nRaw corpus:", comments_df.shape)
print(comments_df.groupby("community_group")["channel_name"].nunique())
print("Saved:", COMMENTS_FILE)
